In [5]:
import os
os.chdir("..")
os.getcwd()

'/Users/sje30/proj/2020/codecheckers/2026'

## main

In [8]:
data_path="../data/clean_fd_civibe_TCS-2e.csv"
meldata_path="../data/clean_saliva_data.csv"
melstat_path="../data/derivatives/hormones/tau_harmonics/melatonin_circadian_parameters.csv"
out_dir="../reports"

In [7]:
# Load data
import pandas as pd

data = pd.read_csv(data_path)
print(data)

FileNotFoundError: [Errno 2] No such file or directory: '../data/clean_fd_civibe_TCS-2e.csv'

In [4]:
# Get and merge melatonin data and melatonin circadian statistiics
from analysis.src.plot_metropsis_group import summarize_contrastsens_by


meldata = pd.read_csv(meldata_path)
melstat = pd.read_csv(melstat_path)
meldata = meldata.merge(melstat, on="participant_id")

# Summarize data across participants
pid_df = summarize_contrastsens_by(
    data=data, group_vars=["participant_id", "block", "condition", "freq"]
)
pid_df.sort_values(by=["participant_id", "condition", "freq", "block"])

ModuleNotFoundError: No module named 'analysis'

In [ ]:
blub = plt.cm.get_cmap("tab20", 11)
costum_cmap = custom_colors = [
    "#1f77b4",  # Blue
    "#ff7f0e",  # Orange
    "#2ca02c",  # Green
    "#d62728",  # Red
    "#9467bd",  # Purple
    "#8c564b",  # Brown
    "#e377c2",  # Pink
    "#7f7f7f",  # Gray
    "#bcbd22",  # Khaki
    "#17becf",  # Teal
]
costum_cmap[1]

In [ ]:
cs_col = "contrast_sensitivity"
avgdata = summarize_contrastsens_by(
    data=data, group_vars=["block", "condition", "freq", "participant_id"], cs_col=cs_col
)
avgdata.sort_values(by=["condition", "freq", "block"])

df2 = avgdata[avgdata["freq"] == 2].pivot_table(
    index=["participant_id", "block"], columns="condition", values="cs_mean"
).reset_index()
df8 = (
    avgdata[avgdata["freq"] == 8]
    .pivot_table(
        index=["participant_id", "block"], columns="condition", values="cs_mean"
    )
    .reset_index()
)

In [ ]:
from .plot_metropsis_group import create_plot_cs_comparison_grid
import numpy as np



# Create average plots
fig, axs = create_plot_cs_comparison_grid(df=avgdata)
fig.set_figheight(10)
fig.set_figwidth(7)
filepath = os.path.join(
    out_dir,
    "fig",
    "{}.pdf".format("img_metropsis_blocks_group"),
)
fig.savefig(filepath)
print("Figure saved in: ", filepath)
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm


def plot_cs_comparison(ax, df, cond1, cond2):
    """
    Creates a scatter plot comparing cs_mean of two conditions for each participant and block.

    Parameters:
        df (pd.DataFrame): DataFrame with columns ['block', 'condition', 'participant_id', 'cs_mean']
        cond1 (str): The first condition to compare
        cond2 (str): The second condition to compare

    Returns:
        fig, ax: The matplotlib figure and axes objects
    """
    condition_dict = {
        "achromatic": "L+M+S (luminance)",
        "red-green": "L-M (red-green)",
        "blue-yellow": "S (blue-yellow)",
    }
    custom_colors = [
        "#ED6D66",
        "#D5842F",
        "#A49930",
        "#5BA930",
        "#52B455",
        "#54B89D",
        "#50B0D7",
        "#459AF8",
        "#AA7AF7",
        "#EB5DE6",
        "#EC5AAD",
    ]

    # Pivot the DataFrame to have conditions as columns
    pivot_df = df.pivot_table(
        index=["participant_id", "block"], columns="condition", values="cs_mean"
    ).reset_index()

    # Create a colour palette for participants
    unique_participants = pivot_df["participant_id"].unique()
    num_participants = len(unique_participants)
    cmap = plt.colormaps["tab20"]  # Get the colormap

    # Plot each participant's data with block transparency
    for i, pid in enumerate(unique_participants):
        pid_data = pivot_df[pivot_df["participant_id"] == pid]

        # Calculate mean and standard deviation
        mean = pid_data[cond1].mean()
        std = pid_data[cond1].std()

        # Create an ellipse representing the data
        x = np.linspace(mean - 3 * std, mean + 3 * std, 100)
        y = norm.pdf(x, loc=mean, scale=std)
        y *= 0.1  # Scale the ellipse to fit the plot

        ax.plot(x, y, color=custom_colors[i], alpha=0.4, label=f"{pid}")

        # Plot the mean as a scatter point
        ax.scatter(mean, mean, color=custom_colors[i], edgecolors="black", s=50)

    # Plot 1:1 line for reference
    max_val = max(pivot_df[cond1].max(), pivot_df[cond2].max())
    min_val = min(pivot_df[cond1].min(), pivot_df[cond2].min())
    ax.plot([0, max_val], [0, max_val], "k--", alpha=0.7)

    # Scale
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(min_val - 0.01, max_val + 0.1)  # max(df["cs_mean"]))
    ax.set_xlim(min_val - 0.01, max_val + 0.1)  # max(df["cs_mean"]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ",")))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ",")))

    # Labels and title
    ax.set_xlabel(f"{condition_dict[cond1]}")
    ax.set_ylabel(f"{condition_dict[cond2]}")
    # fig.tight_layout()
    # fig.show()

    # Legend
    ax.legend(loc="best", bbox_to_anchor=(1.05, 1), title="Participants")

    return ax


def create_plot_comparison_grid(df):
    # Configurations for the subplot grid
    conditions = ["achromatic", "blue-yellow", "red-green"]
    freq = 2

    # Figure dimensions
    nrows = 3
    ncols = 2

    # Create a 3x2 grid of subplots
    distance_left_edge = 0.1
    distance_right_edge = 0.06
    distance_top_edge = 0.1
    distance_bottom_edge = 0.1
    fig, axs = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        # figsize=figsize,
        gridspec_kw={
            "wspace": 0.2,
            "hspace": 0.2,
            "left": distance_left_edge,
            "right": (1 - 2 * distance_right_edge),
            "top": (1 - distance_top_edge),
            "bottom": distance_bottom_edge,
        },
    )

    # Collect legend handles and labels from the first subplot
    handles, labels = None, None

    # Utilize the plot_cs_comparison function for scatter plots
    for row, combo in enumerate(
        [
            ["achromatic", "red-green"],
            ["achromatic", "blue-yellow"],
            ["red-green", "blue-yellow"],
        ]
    ):
        for col, (_, group_by_freq_data) in enumerate(df.groupby("freq")):
            ax = axs[row, col]  # Select the subplot location
            plot_cs_comparison(
                ax=ax, df=group_by_freq_data, cond1=combo[0], cond2=combo[1]
            )

            # Capture handles and labels from first subplot and remove its legend
            if handles is None:  # Only need to capture from first plot with a legend
                handles, labels = ax.get_legend_handles_labels()

            # Always remove the legend from this subplot
            legend = ax.get_legend()
            if legend:
                legend.remove()

    # Add a single legend for the entire figure
    fig.legend(
        handles,
        labels,
        ncol=1,
        loc="right",
        # bbox_to_anchor=(0.5, 0.02),
        frameon=False,
    )

    # General title for the entire figure
    fig.suptitle(
        "Comparison of contrast sensitivities of different post-receptoral mechanisms"
    )
    fig.set_figheight(10)
    fig.set_figwidth(7)
    return fig, axs


fig, axs = create_plot_comparison_grid(df=avgdata)

# MPI Poster Day

Figures generated for MPI Poster Day on April 29 2025.

Initial data loading...

In [ ]:
# Load data
import os

if "data" not in os.listdir():
    print("Old: ", os.getcwd())
    print(os.chdir(".."))
    print("New:", os.getcwd())

import pandas as pd
from analysis.src.plot_metropsis_group import (
    create_plot_cs_comparison_grid,
    create_plot_blocks_individual_timecourses,
    create_plot_blocks_group_timecourses,
    summarize_contrastsens_by
) 


# Load data
data_path = "./data/clean_fd_civibe_TCS-2e.csv"
out_dir = "."
data = pd.read_csv(data_path)

# Summarize data across participants
pid_df = summarize_contrastsens_by(
    data=data, group_vars=["participant_id", "block", "condition", "freq"]
)
pid_df.sort_values(by=["participant_id", "condition", "freq", "block"])


## Plotting Metropsis and Melatonin Data

### Individual time courses of contrast thresholds

- excluding Participant 200 as data is invalid due to change in monitor setting at undefined time points (somewhere beginning to middle of FD protocol)

In [ ]:
# Create plots of individual time courses - first participant subgroup
pid_df = pid_df.loc[(pid_df["participant_id"] > 200)]
fig, _ = create_plot_blocks_individual_timecourses(data=pid_df, letter="A")
fig.set_figheight(20)
fig.set_figwidth(7)
filepath = os.path.join(
    out_dir,
    "fig",
    "{}.pdf".format("img_metropsis_blocks_individual"),
)
fig.savefig(filepath)
print("Figure saved in: ", filepath)

In [ ]:
import os
import pandas as pd

from analysis.src.plot_metropsis_group import (
    create_plot_blocks_group_timecourses,
    summarize_contrastsens_by,
)

In [ ]:
import os
import pandas as pd

from analysis.src.plot_metropsis_group import (
    create_plot_blocks_group_timecourses,
    create_plot_individual_timecourse_melatonin,
    summarize_contrastsens_by,
)

print(os.getcwd())
# Create individual melatonin plots - first participant subgroup
out_dir = "."
meldata_path = "./data/clean_saliva_data.csv"
melstat_path="./data/derivatives/hormones/tau_harmonics/melatonin_circadian_parameters.csv"

melraw = pd.read_csv(meldata_path)
melstat = pd.read_csv(melstat_path)
meldata = melraw.merge(melstat, on="participant_id")

meldata = meldata.loc[(meldata["participant_id"] > 200)]
fig, _ = create_plot_individual_timecourse_melatonin(data=meldata, letter="B")
fig.set_figheight(20)
fig.set_figwidth(4)
filepath = os.path.join(
    out_dir,
    "fig",
    "img_{}{}.pdf".format("melatonin", ""),
)
fig.savefig(filepath)
print("Figure saved in: ", filepath)

In [ ]:
# Create average melatonin time course across participants
fig, _ = create_plot_blocks_group_timecourses(data=meldata, letter="C")
fig.set_figheight(2)
fig.set_figwidth(4)
filepath = os.path.join(
    out_dir,
    "fig",
    "img_{}{}.pdf".format("melatonin", ""),
)
#fig.savefig(filepath)
print("Figure saved in: ", filepath)

In [ ]:
# Create average plots
fig, _ = create_plot_blocks_group_timecourses(data=pid_df)
fig.set_figheight(2)
fig.set_figwidth(7)
filepath = os.path.join(
    out_dir,
    "fig",
    "{}.pdf".format("img_metropsis_blocks_group"),
)
fig.savefig(filepath)
print("Figure saved in: ", filepath)
fig.show()

In [ ]:
# Create cosine wave data for testing purposes
from pymetropsis.simulate.simulate_hormones import create_time_lookups
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker  

lookup = create_time_lookups(standard_wakeup = "2024-01-01 07:00:00")
fd = lookup["fd"]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_violin_matplotlib(data, ax, participant_id, condition, freq):
    """
    Create a violin plot using pure matplotlib for a single participant.

    Parameters:
    -----------
    data : array-like
        Contrast sensitivity values
    ax : matplotlib.axes.Axes
        Axes to plot on
    participant_id : int
        Participant ID for annotation
    condition : str
        Experimental condition
    freq : int
        Frequency
    """
    # Compute kernel density estimation for violin plot
    kernel_stats = plt.violinplot(
        data,
        positions=[0],
        widths=0.8,
        showmeans=False,
        showextrema=False,
        showmedians=False,
    )

    # Customize violin plot colors
    for pc in kernel_stats["bodies"]:
        pc.set_facecolor("skyblue")
        pc.set_edgecolor("blue")
        pc.set_alpha(0.7)

    # Participant ID as text in plot
    ax.annotate(
        text=f"{freq} Hz",
        xy=(0, 0),
        xytext=(0.85, 0.85),
        xycoords="axes fraction",
        textcoords="axes fraction",
    )
    ax.annotate(
        text=f"Participant {participant_id}",
        xy=(0, 0),
        xytext=(0.03, 0.85),
        xycoords="axes fraction",
        textcoords="axes fraction",
    )

    # Axes and labels
    ax.set_yscale("log")
    ax.tick_params(axis="y")
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(0.03, 10)

    # Remove x-ticks
    ax.set_xticks([])


def create_plots_violin_descriptives(df):
    """
    Create violin plots for all conditions and frequencies.

    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe

    Returns:
    --------
    matplotlib.figure.Figure
        The complete figure with subplots
    """
    # Get unique conditions and frequencies
    conditions = sorted(df["condition"].unique())
    frequencies = sorted(df["freq"].unique())

    # Create figure with subplots
    fig, axes = plt.subplots(
        len(conditions), len(frequencies), figsize=(12, 15), squeeze=False
    )

    # Prepare to capture handles and labels
    handles = None

    # Iterate through conditions and frequencies
    for row, condition in enumerate(conditions):
        for col, freq in enumerate(frequencies):
            # Get current axes
            ax = axes[row, col]

            # Filter data for specific condition and frequency
            condition_freq_df = df[
                (df["condition"] == condition) & (df["freq"] == freq)
            ]

            # Get unique participants
            participants = condition_freq_df["participant_id"].unique()

            # Plot violin plots for each participant
            for pid in participants:
                participant_data = condition_freq_df[
                    condition_freq_df["participant_id"] == pid
                ]["contrast_sensitivity"]

                plot_violin_matplotlib(participant_data, ax, pid, condition, freq)

            # Set row and column labels
            if col == 0:
                ax.set_ylabel(condition)
            if row == len(conditions) - 1:
                ax.set_xlabel(f"{freq} Hz")

    # Adjust layout and add overall title
    plt.tight_layout()
    fig.suptitle("Contrast Sensitivity Violin Plots", y=1.02)

    return fig


# Optional: Demonstration of how to use the function
def plot_example(df):
    fig = create_plots_violin_descriptives(df)
    plt.show()

plt.show()